# DTU Chatbot - Retrieval Augmenting Generation on Master’s program specifications and DTU’s courses

## Introduction

Finding information about master’s programs and courses at DTU can be surprisingly difficult for students. Details about study programs are scattered across different parts of the DTU website, and with more than 700 master's courses available, it’s not easy to get a full overview. The current search functionality doesn’t make it much easier — finding and comparing courses often becomes a frustrating and time-consuming task.

### Motivation

Many students struggle to make informed decisions about their studies simply because it’s hard to access the right information quickly. Exploring different study options, checking course prerequisites, and understanding program structures should be straightforward, but today it often isn’t.

To address this, we developed a chatbot that allows users to ask natural language questions and instantly retrieve relevant information about DTU’s programs and courses. The goal is to make the process of finding and understanding study options much simpler, more efficient, and less frustrating. 

The chatbot is deployed and available here: [DTU Chatbot](https://dtu-chatbot.streamlit.app/).

The whole code is available on [GitHub](https://github.com/RasSoender/RAG_DTU).

# Chatbot Architecture Overview

The chatbot is built through a modular pipeline that handles everything from raw data collection to user interaction and response generation. Here's an overview of how it works:

### 1. Data Collection and Preparation
- **Web Scraping:** The raw data is scraped directly from DTU’s official web pages. This includes detailed descriptions of master's programs and individual courses.
- **Preprocessing and Text Normalization:** After scraping, documents are cleaned and preprocessed to standardize formatting, remove noise, and organize the content.
- **Embedding Models:** The cleaned data is then encoded using **multiple embedding models**. Different models are used to better capture the semantic differences between course content and study program descriptions.

### 2. Storage and Retrieval
- **Separate Vector Databases:** 
  - One vector database is dedicated to **courses**.
  - Another vector database is used for **master’s programs**.
- **Retrieval:** When a user submits a query, the system searches the appropriate database based on the query type. Retrieval is based on:
  - **Keyword search** (for exact matching).
  - **Cosine similarity** (for semantic matching using the embeddings).

### 3. User Interaction and Query Handling
- **User Interface:** Students interact with the chatbot through a simple web-based interface where they can ask questions and receive answers.
- **Query Routing:** The system classifies the user's query to determine whether it relates to a course or a study program and routes it to the correct vector database.
- **Short-Term Memory:** A short-term memory component ("Brain") keeps track of previous exchanges in the session to maintain context and improve multi-turn conversations.

### 4. Response Generation
- **LLM (Large Language Model):** Retrieved documents are passed to a large language model, which generates a coherent and natural-sounding response to the user’s query.

### 5. Deployment and Feedback Loop
- **Deployment:** Deployment: The chatbot is deployed to the cloud, ensuring that students can easily access it anytime for support and information.
- **User Feedback:** Students can provide feedback about the relevance and quality of the answers. This feedback is stored in a MongoDB database and can be used for future model fine-tuning and system improvements.

The whole pipeline is illustrated in the following figure:

![Chatbot Architecture](./images/chatbot_schema.jpeg)

## 1. Data Collection and Preparation

### Web Scraping

One of the core components of a Retrieval-Augmented Generation (RAG) is the information its retrieve and base its answers on. The chatbot should be able to answer questions about master's programs and master's courses on DTU, and it was therefore nessecary to gather those information.

No database or data collection of DTU's master's program or courses was publicly accessible to use for the RAG. Therefore webscraping was applied to collect the data nessecary for the project. The two webpages are https://kurser.dtu.dk/ (about courses) and https://sdb.dtu.dk/ (about programmes) contains all the information needed. 

Two different libraries was used for webscraping the two webpages: 
- crawl4ai
- selenium

Crawl4ai is an asynchronous web crawler with allows the user to easily scrape urls and convert into markdown format. This was used for the scraping of the program information since all data in each url was needed and no further specification was needed. 

Initially crawl4ai was attempted to use on retrieving the course information, but this was not possible, because (https://kurser.dtu.dk/) needed authorisation/log-in which crawl4ai was not able to provide. Instead we opted to utilize Selenium, because it waits for the webpages to render and therefore was able to overcome the authorisation problem. The downside of this was the more HTML-specification and longer running-times. 


In [ ]:

# -------------------

# The file used for scraping program information with crawl4ai

# -----------------

import asyncio
from crawl4ai import AsyncWebCrawler
from crawl4ai.async_configs import BrowserConfig, CrawlerRunConfig, CacheMode
from typing import List
import json

def save_to_file(title, result):
    folder = "data/data_study_programmes"
    filename = f"{folder}/{title.replace(' ', '_')}.json"
    with open(filename, "w", encoding="utf-8") as f:
        result_dict = {
            "markdown": result.markdown
        }
        json.dump(result_dict, f, ensure_ascii=False, indent=2)
    print(f"Result saved to {filename}")


async def sequntial_crawl(urls, titles):
    browser_config = BrowserConfig(
        headless=True,
        verbose=True,
        extra_args=["--disable-gpu", "--disable-dev-shm-usage", "--no-sandbox"]
    )

    crawl_config = CrawlerRunConfig(cache_mode=CacheMode.BYPASS)

    crawler = AsyncWebCrawler(config=browser_config)
    print("starting Crawler")
    await crawler.start()

    try:
        for i, url in enumerate(urls): 
            print(f"Crawling {url}...")

            result = await crawler.arun(
                url=url,
                config=crawl_config
            )
            if result.success:
                print(result.url, "crawled OK!")
                save_to_file(titles[i], result)
            else:
                print("Failed", result.url,"-", result.error_message)

    except Exception as e: 
        print(f"Error: {e}")
    finally:
        print("Closing crawler")
        await crawler.close()


if __name__ == "__main__":
    with open("reference_data.json", "r", encoding="utf-8") as file:
        data = json.load(file)

    # Extract the arrays
    study_programme_urls = data.get("study_programme_urls", [])
    study_programme_titles = data.get("study_programme_titles", [])
    
    asyncio.run(sequntial_crawl(study_programme_urls, study_programme_titles))

In [ ]:

# -------------------

# Small codesnippet from the course scraping file. 
# Two functions used for scraping specific HTML elements. 

# -----------------

def scraping_elements_left(driver, j, scraped_info):
    # Get all rows at once to reduce DOM queries
    try:
        rows = driver.find_elements(By.XPATH, f"//div[@class='box information']//table[{j}]//tr")
        for row in rows:
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) >= 2:
                key = cols[0].text.strip()
                value = cols[1].text.strip()
                if key:  # Only add if key exists
                    scraped_info[key] = value
    except Exception as e:
        print(f"Error in scraping_elements_left: {e}")
    return scraped_info

def scraping_elements_right(driver, scraped_info):
    try:
        # Get all section titles at once
        all_keys = []
        keys_elements = driver.find_elements(By.XPATH, "//div[@class='col-md-6 col-sm-12 col-xs-12']//div[@class='box']//div[@class='bar']")
        for element in keys_elements:
            key = element.text.strip()
            if key:
                all_keys.append(key)
        
        # Get the full text content
        all_text = scrape_text(driver, "//div[@class='col-md-6 col-sm-12 col-xs-12']//div[@class='box']")
        
        if all_text and all_keys:
            # Split text by titles
            split_text_by_titles(all_text, all_keys, scraped_info)
    except Exception as e:
        print(f"Error in scraping_elements_right: {e}")
    
    return scraped_info

To further enrich the possible answers and details that the chatbot can give, it was decided to also scrape the webpage https://dtucourseanalyzer.pythonanywhere.com/. This webpage contains information about courses which is either based on student inputs or otherwise not available on the offical DTU webpage (i.e. workload burden, avg. rating, failed students, avg. grade, etc.). 

Selenium was utilized, because only specific information was necesserary which could be easilier specified with Selenium. A code snippet is shown below: 

In [ ]:
def scraping_elements(driver, scraped_info):
    xpath = f"//div[@class='container']//div[@class='col']//li[@class='list-group-item'][1]//tr[5]//td"
    signups = scrape_text(driver, xpath)
    scraped_info["signups"] = signups

    xpath = f"//div[@class='container']//div[@class='col']//li[@class='list-group-item'][2]//table[2]//tr[1]//td"
    average_grade = scrape_text(driver, xpath)
    scraped_info["average grade"] = average_grade

    xpath = f"//div[@class='container']//div[@class='col']//li[@class='list-group-item'][2]//table[2]//tr[2]//td"
    failed_students = scrape_text(driver, xpath)
    scraped_info["failed students in percent"] = failed_students

    xpath = f"//div[@class='container']//div[@class='col']//li[@class='list-group-item'][3]//tr[1]//td"
    workload_burden = scrape_text(driver, xpath)
    scraped_info["workload burden"] = workload_burden

    xpath = f"//div[@class='container']//div[@class='col']//li[@class='list-group-item'][3]//tr[2]//td"
    overworked_students = scrape_text(driver, xpath)
    scraped_info["overworked students in percent"] = overworked_students

    xpath = f"//div[@class='container']//div[@class='col']//li[@class='list-group-item'][4]//tr[1]//td"
    average_rating = scrape_text(driver, xpath)
    scraped_info["average rating"] = average_rating

    return scraped_info

def process_url(url, index):
    try:
        print(f"Processing URL {index}")
        driver = setup_driver(headless=True)
        
        # Set timeout to avoid hanging on problematic pages
        driver.set_page_load_timeout(30)
        
        try:
            driver.get(url)
        except Exception as e:
            print(f"Error loading URL {url}: {e}")
            driver.quit()
            return False
            
        # Refreshing the page
        driver.refresh()
        time.sleep(1)  # Reduced sleep time
        
        scraped_info = dict()
        scraped_info["course title"] = scrape_text(driver, "//h5")
        
        if not scraped_info["course title"]:
            print(f"Failed to get course title for URL {url}")
            driver.quit()
            return False

        scraped_info = scraping_elements(driver, scraped_info)

        # Save results
        saving_into_json(scraped_info)
        
        driver.quit()
        return True
    except Exception as e:
        print(f"Error processing URL {url}: {e}")
        return False

Even though the webscraping is slow, it only need to be done one time (or possible more if some information are updated). 

A lot of other small details as collecting the urls, cleaning the scraped data, converting to json-format, merging files etc. was also done, but did seem unnecessary and nonessential to include in this technical overview. 

### Preprocessing and Text Normalization

Before indexing the data for retrieval, we applied a structured preprocessing and text normalization pipeline to both the course and master's programme content.

For the course data, the first step involved cleaning and organizing the raw information. We standardized field names, removed irrelevant metadata, and ensured that text fields were consistently formatted. The normalization function first cleaned spaces and punctuation:



In [ ]:
def clean_text(text):
    text = re.sub(r'\s+([.,:%])', r'\1', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

We also parsed and enriched the data, for example, automatically detecting the semester(s) a course belonged to by analyzing the schedule description:

In [ ]:
def add_semester_info(entry):
    schedule = entry.get("Schedule", "").lower()
    semesters = []
    if "spring" in schedule:
        semesters.append("13 weeks spring")
    if "autumn" in schedule:
        semesters.append("13 weeks autumn")
    for month in ["january", "august", "july", "june"]:
        if month in schedule:
            semesters.append(f"3 weeks {month}")
            break
    entry["Semester"] = semesters if semesters else ["Unknown"]
    return entry

Exam and re-exam dates were attached to each course entry based on mappings available through course codes or detected keywords:

In [ ]:
def attach_exam_dates(entry, course_code, exam_map, reexam_map):
    # Extract schedule codes and link to exam dates
    ...
    entry["Exam"] = exam
    entry["Re_exam"] = reexam
    return entry

After normalization, all textual fields were combined into a single preprocessed text string, and optional stemming or lemmatization was applied:

In [ ]:
def preprocess_course_text(course_dict, do_stemming=False, do_lemmatization=False):
    text = " ".join(str(v) for v in course_dict.values() if isinstance(v, str))
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    tokens = word_tokenize(text)
    if do_stemming:
        tokens = [stemmer.stem(word) for word in tokens]
    elif do_lemmatization:
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
    processed_text = " ".join(tokens)
    return processed_text, len(tokens)


For the master's programmes, the pipeline was slightly different. After cleaning the raw data and renaming fields, we merged relevant sections like programme provision, specializations, and curriculum into a single consolidated field:

In [ ]:
def merge_curriculum_info(data):
    keys_to_merge = ["programme_provision", "specializations", "curriculum"]
    merged_text = "\n".join([data.get(key, "").strip() for key in keys_to_merge if data.get(key, "")])
    data["curriculum_info"] = merged_text
    for key in keys_to_merge:
        data.pop(key, None)
    return data

Special formatting artifacts, such as markdown links, were removed from the text during cleaning:

In [ ]:
def clean_text(text):
    text = re.sub(r"\[([^\]]+)\]\([^\)]+\)", r"\1", text)  # Remove markdown links
    text = text.replace("\n", " ").replace("*", "")
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


When a section exceeded a certain length (more than 800 tokens), it was summarized using a language model to maintain retrieval efficiency:

In [ ]:
def summarization(text, model="gpt-4o-mini-2024-07-18"):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": f"Summarize the following text for retrieval: {text}"}],
        max_tokens=500,
        temperature=0.5
    )
    summary = response.choices[0].message.content
    return summary


### Embedding Generation

Once the text data for courses and master’s programmes was preprocessed and normalized, we generated vector embeddings to enable semantic search and retrieval. Different strategies were applied for courses and programmes, but both relied on a combination of a general-purpose embedding model and a specialized model for names and short texts.

#### Course Embeddings
For each course, we generated two separate types of embeddings: one for the full course content and one specifically for the course name.

The full course content was embedded using OpenAI's `text-embedding-3-small` model, which produced dense, high-quality representations suitable for semantic retrieval:

In [ ]:
def get_embedding(text, model="text-embedding-3-small"):
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding

In parallel, the course names were embedded using a lighter and faster model optimized for short texts, `paraphrase-MiniLM-L6-v2` from the Sentence Transformers library:

In [ ]:
name_embedding_model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

def get_name_embedding(text):
    embedding = name_embedding_model.encode(text)
    return embedding.tolist()

During processing, both embeddings were computed and stored along with course metadata. The full course description and course name embeddings were then combined into a structured dictionary:

In [ ]:
def process_courses(json_file_path, use_postprocessed=True):
    with open(json_file_path, "r", encoding="utf-8") as f:
        courses = json.load(f)

    embeddings = {}
    for course_id, data in courses.items():
        text = data["postprocessed_course"] if use_postprocessed else data["preprocessed_course"]
        course_name = data.get("metadata", {}).get("title", f"Course {course_id}")
        course_code = data.get("metadata", {}).get("course_code", "")
        enhanced_name = f"{course_name} (Code: {course_code})" if course_code else course_name

        content_embedding = get_embedding(text)
        name_embedding = get_name_embedding(enhanced_name)

        embeddings[course_id] = {
            "content_embedding": content_embedding,
            "name_embedding": name_embedding,
            "metadata": data.get("metadata", {})
        }
    return embeddings


The resulting embeddings were saved for later retrieval and querying.

#### Master's Programme Embeddings

For master’s programmes, the structure of the data was more complex and required a chunk-based approach. Each programme was divided into meaningful sections (e.g., *Programme Specification*, *Admission Requirements*, *Curriculum*), and embeddings were created for each individual section.

First, the processed and summarized data were merged to ensure we had a complete but concise version of each programme:

In [ ]:
def merge_files(normal, summarized):
    merged = {}
    for programme_name, data in summarized.items():
        merged[programme_name] = {}
        for key, value in data.items():
            if not value or value.strip() == "":
                merged[programme_name][key] = normal.get(programme_name, {}).get(key, "")
            else:
                merged[programme_name][key] = value
        merged[programme_name]["metadata"] = normal.get(programme_name, {}).get("metadata", {})
    return merged

Then, for each section (or "chunk"), we generated two embeddings: one for the field name and one for the field content. The embedding of the field name used the same `paraphrase-MiniLM-L6-v2` model for short texts, while the field content was embedded using OpenAI’s `text-embedding-3-small` model:

In [ ]:
def process_programme_chunks(json_file_path, json_summarized_path):
    with open(json_file_path, "r", encoding="utf-8") as f:
        programmes = json.load(f)
    with open(json_summarized_path, "r", encoding="utf-8") as f:
        summarized_programmes = json.load(f)

    merged_programmes = merge_files(programmes, summarized_programmes)
    chunked_embeddings = {}
    chunk_counter = 0
    
    for programme_name, data in merged_programmes.items():
        metadata = data.get("metadata", {})
        section_names = metadata.get("section_names", [])
        url = metadata.get("url", "")

        for idx, (field_name, field_content) in enumerate(data.items()):
            if not isinstance(field_content, str) or field_content.strip() == "" or field_name == "programme_name":
                continue

            chunk_id = f"chunk_{chunk_counter}"
            chunk_counter += 1

            chunk_entry = {
                "name_embedding": get_name_embedding(field_name),
                "content_embedding": get_embedding(field_content),
                "metadata": {
                    "course_name": programme_name,
                    "chunk_name": field_name,
                    "url": f"{url}#{section_names[idx].replace(' ', '_')}" if url and section_names else "",
                }
            }
            chunked_embeddings[chunk_id] = chunk_entry
    return chunked_embeddings


Each programme was therefore broken down into manageable pieces, enabling the chatbot to retrieve very specific information from a programme without having to search through the entire document.

By using multiple embedding models and carefully structuring the data at different levels (course, programme, chunk), we were able to maximize retrieval precision and minimize irrelevant or unfocused results during user interaction.

## 2. Storage and Retrieval

Once embeddings were generated, we designed a storage and retrieval system to make the information easily accessible through natural language queries. This system is based on two separate vector databases: one for courses and one for master’s programmes. We use Weaviate as the vector database, chosen for its hybrid search capabilities, scalability, and support for named vectors.

### Vector Databases
We created two separate collections inside Weaviate: a Course collection and a Chunk collection (for Master's programmes).

#### Course collection
Each course was stored as a single object containing metadata, a content embedding (full course description), and a name embedding (course title). The schema included fields such as course code, course name, semester, exam dates, ECTS points, and others. Before inserting the objects, we defined the schema and manually specified two named vectors: `content_embedding` and `name_embedding`. This allowed us to control how content and names were used during retrieval.

The schema definition for courses was created as follows:

In [ ]:
course_collection = weaviate_client.collections.create(
    name="Course",
    description="A university course with detailed information",
    vectorizer_config=[
        Configure.NamedVectors.none(name="content_embedding"),
        Configure.NamedVectors.none(name="name_embedding")
    ],
    properties=[...]
)

After creating the schema, courses were imported into the database with both embeddings and full metadata attached.

#### Chunk Collection (for Programmes):
For master's programmes, each programme was divided into meaningful sections ("chunks"), such as programme specifications, admission requirements, curriculum, etc. Each chunk was stored separately, with its own embeddings and metadata linking it back to the original programme.

The chunk schema was similarly defined with two named vectors:

In [ ]:
chunk_collection = weaviate_client.collections.create(
    name="Chunk",
    description="A chunk of course content",
    vectorizer_config=[
        Configure.NamedVectors.none(name="content_embedding"),
        Configure.NamedVectors.none(name="name_embedding")
    ],
    properties=[...]
)


This chunked structure made it possible to retrieve very specific parts of a programme without retrieving the entire programme document.

Both collections were hosted on Weaviate Cloud, allowing for fast hybrid searches combining both keyword and semantic search strategies.

### Retrieval
Retrieval was designed to be efficient and flexible, adapting the strategy depending on whether the query targeted courses or master's programmes.

For **courses**, when a user submits a query, the system first attempts an exact match based on a 5-digit course code if present. If no such code is detected, it performs a hybrid search that combines semantic search using two distinct embeddings (`content_embedding` and `name_embedding`) with keyword search powered by BM25. The embeddings for the user query are computed at runtime, and similarity is calculated using customizable weights that allow prioritizing the course content.

In [ ]:
response = course_collection.query.hybrid(
    query=query,
    alpha=0.75,
    vector={
        "content_embedding": query_embedding_openai,
        "name_embedding": name_embedding_trans
    },
    limit=top_k,
    target_vector=TargetVectors.manual_weights({"content_embedding": 70, "name_embedding": 30}),
    return_metadata=MetadataQuery(distance=True)
)


For **programmes**, retrieval operates in a similar way but at the chunk level. Each section of a programme page, such as admission requirements or curriculum, is embedded and indexed independently. This enables the system to retrieve only the specific portion of the programme that matches the user's query. As with courses, the system uses a hybrid of semantic similarity based on dual embeddings and keyword matching via BM25.

In [ ]:
response = chunk_collection.query.hybrid(
    query=query,
    alpha=0.75,
    vector={
        "content_embedding": content_embedding,
        "name_embedding": name_embedding
    },
    limit=top_k,
    target_vector=TargetVectors.manual_weights({"content_embedding": 70, "name_embedding": 30}),
    return_metadata=MetadataQuery(distance=True)
)

This two-level hybrid retrieval system allowed us to achieve both high precision (via semantic similarity) and high recall (via keyword search), delivering accurate and relevant results to the users' natural language queries.

## 3. User Interaction and Query Handling

### User Interface

### Query Routing

Every incoming query is analyzed by the `QueryRouter` to determine its intent and corresponding handling strategy. This analysis is powered by an LLM that inspects the user input, recent conversation history, and available Master's programmes. The LLM outputs a structured JSON object that categorizes the query into one of four types:

- `course_query`: the query relates to a specific course.

- `programme_query`: the query targets a Master's programme.

- `conversation_query`: the user refers to previous exchanges without re-specifying the context.

- `unknown_query`: the intent is ambiguous or unsupported.

The analysis also flags whether past conversation history is required to resolve the query.

Once classified, the router executes the appropriate logic for retrieving relevant information and preparing the context for LLM response generation. The logic is summarized in the following snippet:


In [ ]:
if query_type == self.QUERY_TYPES["COURSE"]:
    search_results = self.vector_db_client.search_courses(normalized_query, filters=filters)
elif query_type == self.QUERY_TYPES["PROGRAMME"]:
    search_results = self.vector_db_client.search_programmes(query, filter_programme=prog_name)
elif query_type == self.QUERY_TYPES["CONVERSATION"]:
    response = self._generate_response(normalized_query, previous_query_type, requires_history, history_context, context_str)


If the query is related to a course or programme, the system performs a hybrid vector search via Weaviate using both `content_embedding` and `name_embedding`. The retrieved documents are then formatted and passed to the LLM for generation of a natural-language answer. 

For `conversation_query` types, the router assumes that the user is referring to the most recent explicit query context — either a course or a Master's programme. If the last query was a course-related one, the previously retrieved course data is reused, as the course of interest was likely already identified. In contrast, if the last query involved a Master's programme, the router performs a fresh search using the current query text to locate the relevant programme chunks, since the specific section of interest may differ from before. This strategy ensures continuity in conversation while maintaining accuracy and relevance.

The final LLM response is structured, markdown-formatted, and always includes a verifiable source link pointing to the official DTU course or programme page.

### Short-Term Memory

Two memory modules ensure context awareness in multi-turn conversations:

- `QueryMemory`: stores simplified metadata about each query and classification result.

- `Memory`: stores detailed retrieved content for programme or course data.

Each memory entry includes timestamps, query type, user question, retrieved results, and optionally prior information used.

In [ ]:
@dataclass
class MemoryItem:
    timestamp: float
    user_query: str
    query_type: str
    retrieved_info: str
    used_before_info: str = None
    metadata: Dict[str, Any] = None


Memory entries are added after every interaction:

In [ ]:
self.query_memory.add(
    user_query=query,
    query_type=query_type,
    additional_info=query_information,
    used_before_info=other_info
)

self.context_memory.add(
    user_query=query,
    query_type=query_type,
    retrieved_info=context_information,
    used_before_info=other_info
)

To build history for the prompt, a formatted string is generated:

In [ ]:
def get_formatted_history(self, relevant_items: List[MemoryItem] = None) -> str:
    ...
    history_parts.append(
        f"User: {item.user_query}\n"
        f"Type: {item.query_type}\n"
        f"Used Before Information: {item.used_before_info}\n"
        f"Retrieved Information:\n{item.retrieved_info}\n"
    )


## 4. Response Generation

### LLM (Large Language Model)

Each user query is answered using a dedicated LLM-based prompt that integrates both search results and short-term memory context. The system never replies directly from the vector database, instead, the retrieved data is formatted into a prompt and sent to the LLM for natural-language generation.

#### Prompt construction
When a query is routed (either as a course query, programme query, or conversation query), the system dynamically builds a detailed prompt that contains:

- The user's latest query

- Whether conversation history is needed

- Retrieved structured information (e.g., course metadata or master's programme chunks)

- Formatted conversation history (up to the last 5 turns)

This is assembled using the following code structure:

In [ ]:
response = self._generate_response(
    query=normalized_query,
    query_type=query_type,
    requires_history=requires_history,
    history_context=history_context,
    retrieved_info=context_str
)
answer = response.get("response", "")
other_info = response.get("other", None)

#### Example Prompt Used
The typical prompt assembled by the system for a course query follows this structure:

In [ ]:
prompt = f"""
You are a highly knowledgeable and polite academic assistant with expertise in all DTU courses...

User Query:
{query}

Requires History: 
{requires_history}

Course Information:
{retrieved_info}

Conversation History:
{history_context}
"""

The fields `{query}`, `{requires_history}`, `{retrieved_info}`, and `{history_context}` are dynamically populated based on the user input and the retrieved context.

#### Response Output
The LLM always responds in a structured JSON format, for example:

In [ ]:
{
  "response": "**📝 Exam Information for Introduction to Machine Learning (02450):**\n\n\
               The exam is scheduled for **December 15th, 2024**. 📅\
               Make sure you also review the full course page for any updates! \n\n\
               The information has been retrieved from the official course page here: [Course Page](https://kurser.dtu.dk/course/02450)",
  "other": "02450"
}

This design ensures:

- Clear and structured answers

- Markdown formatting with emojis, bold highlights, and links

- Always a source link at the end (course or master's programme page)

- Correct tracking of the main course code or programme name that the assistant used

## 5. Deployment and Feedback Loop

### Streamlit

Streamlit was chosen as the framework for the chatbot’s user interface and deployment following a recommendation from *Simon Soerensen* at 2021.ai, who shared his experience with the tool.

The Streamlit framework has many built-in methods and functions that enable easy development of functionalities for a chatbot. 
The following code shows how the main chat receives the prompt and generates a response.
To ensure continous interaction the `st.chat_input()` has been utilized. Its ensures that the script rerun such that the user can continue to prompt with the chatbot.  
Prompts and responses are saved into `st.session_state.messages()`, which maintains the conversation history across reruns and enables the display of past interactions in a continuous chat format.

In [ ]:
prompt = st.chat_input("What can I help with?")
if prompt:
    # Generate a unique ID for this message
    message_id = str(uuid.uuid4())
    
    with st.chat_message("user"):
        st.markdown(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt, "message_id": message_id})

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            try:
                if st.session_state.active_program_internal:
                    response, _ = router.route_query(prompt, st.session_state.active_program_internal)
                else:
                    response, _ = router.route_query(prompt)
                st.markdown(response)
                
                # Generate a unique ID for the response
                response_id = str(uuid.uuid4())
                st.session_state.messages.append({"role": "assistant", "content": response, "message_id": response_id})
                
                # Force a rerun to show the rating widget immediately
                st.rerun()
                
            except Exception as e:
                response = f"❌ Error: {e}"
                traceback.print_exc()
                st.error(response)
                st.session_state.messages.append({"role": "assistant", "content": response, "message_id": str(uuid.uuid4())})



#### Deployment to cloud

Streamlit Cloud enables free creation, deployment, and management of Streamlit apps. Developers can push their app code to a GitHub repository and connect it directly to Streamlit Cloud for continuous deployment. Once deployed, the chatbot becomes accessible via a public URL, allowing students and users to interact with it from any device with internet access. 



### Feedback Loop

To monitor the quality of responses and gather user feedback, a MongoDB database has been employed. This database stores user ratings and opinions associated with each chatbot response, enabling ongoing evaluation and improvement of the chatbot’s performance. 

The following snippet of the save_rating function shows how user feedback on chatbot responses is structured and stored in a MongoDB database. Each rating includes metadata such as session and message IDs, the original query and response, the user's rating and optional comment, the active program context, and a timestamp.

In [ ]:
# ....
# ....
# ....

# Create rating document
    rating_doc = {
        "rating_id": rating_id,
        "session_id": st.session_state.session_id,
        "message_id": message_id,
        "query": query,
        "response": response,
        "rating": rating,
        "comment": comment,
        "program": program,
        "timestamp": timestamp
    }
    
    # Save to MongoDB
    try:
        # Get MongoDB connection
        mongo_client = get_mongodb_connection()
        if mongo_client:
            db = mongo_client.get_database("dtu_feedback")
            ratings_collection = db.get_collection("chat_ratings")
            
            # Insert the rating document
            ratings_collection.insert_one(rating_doc)

# ....
# ....
# ....
